# Noctua — Readiness Forecaster → ExecuTorch (.pte)

End-to-end export of the tiny MLP that predicts **tomorrow's Oura readiness score**
from 8 normalized on-device features (`WellnessFeatures.toVector()` in `noctua-ai`).

**Model contract:** input `float32 [1, 8]` → output `float32 [1, 1]` (readiness 0–100).

Steps: 1) define → 2) warm-start on the heuristic prior → 3) export via
`torch.export → to_edge → to_executorch` → 4) validate the `.pte` with the ExecuTorch runtime.

For production, replace step 2 with fine-tuning on the user's own history — the contract never changes.

In [ ]:
# pip install torch executorch
import torch
import torch.nn as nn

FEATURES = 8  # WellnessFeatures.VECTOR_SIZE — keep in sync with noctua-ai
torch.manual_seed(7)

## 1. Model definition

8 → 32 → 16 → 1 MLP. Small on purpose: <5 ms inference and a few KB of weights on any modern phone.

In [ ]:
class ReadinessForecasterNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEATURES, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = ReadinessForecasterNet().eval()
print(sum(p.numel() for p in model.parameters()), 'parameters')

## 2. Warm-start on the heuristic prior

Synthetic targets mirror `LinearHeuristicForecaster`: base 78, sleep debt hurts (−14),
HRV z-score helps (+10), temperature deviation hurts (−12), etc. This teaches the network
physiologically sensible *directions*; personal fine-tuning replaces the magnitudes.

In [ ]:
x = torch.rand(2048, FEATURES) * 2 - 1
w = torch.tensor([-14.0, 10.0, 8.0, 12.0, 8.0, -12.0, 6.0, 3.0])
y = (78.0 + x @ w + torch.randn(2048) * 2.0).unsqueeze(1)

opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for epoch in range(1500):
    opt.zero_grad()
    loss = nn.functional.mse_loss(model(x), y)
    loss.backward()
    opt.step()
    if epoch % 300 == 0:
        print(f'epoch {epoch:4d}  mse {loss.item():.3f}')
print(f'final warm-start loss: {loss.item():.3f}')

## 3. Export to ExecuTorch

In [ ]:
from executorch.exir import to_edge
from torch.export import export as torch_export

aten = torch_export(model, (torch.rand(1, FEATURES),))
program = to_edge(aten).to_executorch()
with open('readiness_forecaster.pte', 'wb') as f:
    f.write(program.buffer)

import os
print('wrote readiness_forecaster.pte —', os.path.getsize('readiness_forecaster.pte'), 'bytes')

## 4. Validate the .pte with the ExecuTorch runtime

Same runtime class the Android app uses (`ExecuTorchForecaster` in `noctua-ai`),
so a green check here means a green check on-device. Expect sensible ordering:
strained input < neutral input < strong input.

In [ ]:
from executorch.runtime import Runtime

rt = Runtime.get()
prog = rt.load_program('readiness_forecaster.pte')
method = prog.load_method('forward')

for label, t in [('neutral', torch.zeros(1, FEATURES)),
                 ('strong ', torch.full((1, FEATURES), 0.5)),
                 ('strain ', torch.full((1, FEATURES), -0.5))]:
    out = method.execute([t])
    val = out[0].item() if hasattr(out[0], 'item') else float(out[0][0])
    print(f'{label} -> predicted readiness {val:.1f}')
print('PTE VALIDATION OK')